# Step 4 - claim extraction, taxonomy and programmatic pre-checks (pilot)

Input: the Step 3 generations (every model x condition) plus the hidden Stockfish reference of the pilot positions.
Output (`outputs/step4_claims/<month>/`): one row per atomic claim with its category (Hebbar-style taxonomy), the rule that assigned it,
an automatic **proposed** verdict where python-chess can decide, and blind annotation sheets for the manual work.

Everything is deterministic: rule-based, no LLM call, no randomness except seeded shuffles. The logic lives in `claims_lib.py` (hashed into the manifest); this notebook only does validation, I/O and reporting.
Save the notebook (Ctrl+S) before **Run All** so the recorded source hash matches the executed code. Commit before the run so the manifest shows a clean state.

In [1]:
import datetime, hashlib, json, math, platform, random, re, subprocess, sys
from importlib import metadata
from pathlib import Path
import pandas as pd, chess
import config, claims_lib as L

PROJECT = Path.cwd()
MONTH = config.MONTH
MODELS = list(getattr(config, "LLM_PILOT_MODELS", []))
CONDITIONS = list(getattr(config, "LLM_PILOT_CONDITIONS", ["A", "B", "C"]))
REPEATS = int(getattr(config, "LLM_PILOT_REPEATS_PER_CONDITION", 1))
P1 = config.OUT_DIR / "step1_pilot" / MONTH
SF = config.OUT_DIR / "stockfish_recheck" / MONTH
S3 = config.OUT_DIR / "step3_llm_pilot" / MONTH
OUT = config.OUT_DIR / "step4_claims" / MONTH
OUT.mkdir(parents=True, exist_ok=True)
AUDIT_N = 12                        # explanations in the extraction audit sheet (checked by hand)
KEY = ["model", "condition", "pilot_id", "repeat"]
problems = []                       # everything flagged below ends up in the manifest


def slug(m): return re.sub(r"[^A-Za-z0-9._-]", "_", m)
def warn(msg): problems.append(msg); print("FLAG", msg)
def sha256_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def csv_bytes(df): return df.to_csv(index=False, lineterminator="\n", float_format="%.10g").encode("utf-8")


def git(*a):
    try: return subprocess.run(["git", "-C", str(PROJECT), *a], check=True, capture_output=True, text=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError): return None


def notebook_code_sha256():
    nb = json.loads((PROJECT / "step4_claims.ipynb").read_text(encoding="utf-8"))
    return hashlib.sha256("\n".join("".join(c["source"]) for c in nb["cells"] if "step4" in c.get("metadata", {}).get("tags", [])).encode()).hexdigest()


print(f"month {MONTH} | models {MODELS} | conditions {CONDITIONS} | repeats {REPEATS}")

month 2026-10 | models ['qwen2.5:7b', 'deepseek-r1:8b', 'phi4-mini:latest', 'llama3.1:8b'] | conditions ['A', 'B', 'C'] | repeats 1


## 1. Verify the Step 3 inputs (hash chain and CPU settings)

In [2]:
pilot_manifest = json.loads((P1 / "pilot_manifest.json").read_text(encoding="utf-8"))
pilot_for_llm = pd.read_csv(P1 / "pilot_for_llm.csv", dtype=str)
pilot_key = pd.read_csv(P1 / "pilot_key.csv", dtype={"pilot_id": str, "position_id": str})
if sha256_file(P1 / "pilot_for_llm.csv") != pilot_manifest["llm_input_sha256"]:
    warn("pilot_for_llm.csv differs from pilot_manifest.json")

rows = []
for m in MODELS:
    d = S3 / slug(m)
    man = json.loads((d / "run_manifest.json").read_text(encoding="utf-8"))
    raw = (d / "generations.jsonl").read_bytes()
    gen_ok = man.get("generations_sha256") in (hashlib.sha256(raw).hexdigest(), hashlib.sha256(raw.replace(b"\r\n", b"\n")).hexdigest())
    o = man["settings"]["options"]
    prefix = getattr(config, "LLM_PILOT_MODEL_ID_PREFIXES", {}).get(m)
    rows.append(dict(model=m, digest=man["model_digest"][:12], digest_ok=(prefix is None or man["model_digest"].startswith(prefix)),
                     ollama=man["ollama_version"], num_gpu=o["num_gpu"], threads=o["num_thread"], temp=o["temperature"], seed=o["seed"],
                     num_ctx=o["num_ctx"], num_predict=o["num_predict"], repeats=man["repeats_per_condition"],
                     done=f'{man["completed_generations"]}/{man["expected_generations"]}', generations_hash_ok=gen_ok,
                     input_ok=man["settings"]["input_csv_sha256"] == pilot_manifest["llm_input_sha256"], clean_git=not man["git_uncommitted_changes"]))
env = pd.DataFrame(rows)
print(env.to_string(index=False))
if (env.num_gpu != 0).any(): warn("a model was not run CPU-only (num_gpu != 0)")
for col in ("threads", "ollama", "seed", "temp"):
    if env[col].nunique() > 1: warn(f"{col} differs between models: {env[col].tolist()}")
if not env.digest_ok.all(): warn("a model digest differs from config.LLM_PILOT_MODEL_ID_PREFIXES")
if not env.input_ok.all(): warn("a Step 3 run used a different pilot_for_llm.csv")
if not env.generations_hash_ok.all(): warn("generations.jsonl differs from the hash in its run_manifest.json (or the hash method is different)")
if (env.repeats < 2).all(): warn("only 1 repeat per condition: run-to-run determinism on CPU is NOT yet tested")

           model       digest  digest_ok  ollama  num_gpu  threads  temp  seed  num_ctx  num_predict  repeats  done  generations_hash_ok  input_ok  clean_git
      qwen2.5:7b 845dbda0ea48       True 0.40.16        0        8   0.0    46     2048          400        1 30/30                 True      True      False
  deepseek-r1:8b 6995872bfe4c       True 0.40.16        0        8   0.0    46     4096         3000        1 30/30                 True      True      False
phi4-mini:latest 78fad5d182a7       True 0.40.16        0        8   0.0    46     2048          400        1 30/30                 True      True      False
     llama3.1:8b 46e0c10c039e       True 0.40.16        0        8   0.0    46     2048          400        1 30/30                 True      True      False
FLAG only 1 repeat per condition: run-to-run determinism on CPU is NOT yet tested


## 2. Load the generations and run quality control

In [3]:
gen = pd.DataFrame([json.loads(l) for m in MODELS for l in (S3 / slug(m) / "generations.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()])
if gen.duplicated(KEY).any(): raise ValueError("duplicate (model, condition, pilot_id, repeat) rows in the generations")
expected = len(MODELS) * len(CONDITIONS) * len(pilot_for_llm) * REPEATS
if len(gen) != expected: warn(f"{len(gen)} generations found, {expected} expected")
gen["words"] = gen.text.str.split().str.len()
gen["truncated"], gen["think_unterminated"] = gen.truncated.astype(bool), gen.think_unterminated.astype(bool)
gen["usable"] = ~gen.truncated & ~gen.think_unterminated & gen.text.str.strip().ne("")

# invariants of the experimental design
a = gen.loc[gen.condition == "A", "prompt"]
if a.str.contains(r"stockfish|centipawn|\bcp\b|principal|engine|evaluation", case=False, regex=True).any():
    warn("a Condition A prompt contains engine information")
if (gen.groupby(["condition", "pilot_id"]).prompt.nunique() > 1).any():
    warn("different models received different prompts for the same position and condition")
for c in CONDITIONS[1:]:
    print(f"--- sample Condition {c} prompt ---\n{gen.loc[gen.condition == c, 'prompt'].iloc[0][:700]}\n")

qc = gen.groupby(["model", "condition"]).agg(n=("usable", "size"), usable=("usable", "sum"), truncated=("truncated", "sum"),
        unterminated_think=("think_unterminated", "sum"), mean_words=("words", "mean"), mean_seconds=("seconds", "mean")).round(1).reset_index()
print(qc.to_string(index=False))
excluded = gen.loc[~gen.usable, KEY + ["done_reason", "words", "truncated", "think_unterminated"]]
if len(excluded): warn(f"{len(excluded)} unusable generation(s) excluded from claim extraction (see excluded_generations.csv)")

--- sample Condition B prompt ---
Chess position (FEN): 4rrk1/1p3pp1/2p2b1p/p1Pn4/4b3/P3PN1P/1P1B1KP1/1BRR4 w - - 5 25
White to move.
The move played by the human is Bxe4 (standard algebraic notation).

Explain why a human might choose this move. Describe what it does and what it changes in the position. Use 3 to 5 sentences and refer to specific pieces and squares.
Stockfish evaluation before the candidate move (White perspective; positive favors White): -15 cp.
Stockfish evaluation after the candidate move (White perspective; positive favors White): -139 cp.


--- sample Condition C prompt ---
Chess position (FEN): 4rrk1/1p3pp1/2p2b1p/p1Pn4/4b3/P3PN1P/1P1B1KP1/1BRR4 w - - 5 25
White to move.
The move played by the human is Bxe4 (standard algebraic notation).

Explain why a human might choose this move. Describe what it does and what it changes in the position. Use 3 to 5 sentences and refer to specific pieces and squares.
Stockfish evaluation before the candidate move (White perspect

## 3. Reference data of the pilot positions (hidden labels, used only for the pre-checks)

In [4]:
sf = pd.read_csv(SF / "positions_stockfish.csv", dtype={"position_id": str})
ref = (pilot_for_llm.merge(pilot_key[["pilot_id", "position_id", "kind", "stockfish_cp_loss", "engine_best_san"]], on="pilot_id", how="left")
       .merge(sf[["position_id", "fen", "eval_before_cp", "eval_after_cp"]].rename(columns={"fen": "fen_sf"}), on="position_id", how="left"))
assert len(ref) == len(pilot_for_llm) and ref.eval_before_cp.notna().all(), "pilot positions not found in positions_stockfish.csv"
assert (ref.fen == ref.fen_sf).all(), "FEN differs between pilot_for_llm.csv and positions_stockfish.csv"
ref = ref.set_index("pilot_id")
print(ref[["kind", "move_to_explain_san", "stockfish_cp_loss", "eval_before_cp", "eval_after_cp"]].to_string())

          kind move_to_explain_san  stockfish_cp_loss  eval_before_cp  eval_after_cp
pilot_id                                                                            
P001       bad                Bxe4                124             -15           -139
P002      good                 Bg6                  0            -329           -356
P003       bad                  g5                467             254            721
P004       bad                 Rc1                131            -473           -604
P005       bad                 Nd7                213            -246            -33
P006      good                Rxf4                  0            -288           -288
P007       bad                 Bh3                144               7            151
P008      good                 Rb1                 25              25              0
P009      good                Rxf2                  0               0              0
P010      good                Rfd8                  0            

## 4. Extract claims, classify, pre-check

Pipeline per usable explanation: split into sentences, then atomic clauses; assign one primary category (precedence Legality > Tactical > Quality > Board > Other;
`also_matches` / `review_category` mark ambiguous cases); flag hedged and strategic claims; run the python-chess pre-checks. **Pre-check verdicts are proposals, not labels.**

In [5]:
def route(r):
    if r.auto_verdict != "unknown": return "engine_label" if r.auto_rule == "polarity_vs_label" else "programmatic"
    if r.is_strategic: return "strategic_unverifiable"
    return "engine_manual" if r.category in ("Quality evaluation", "Tactical reasoning") else "manual"


def run_extraction(df):
    out = []
    for g in df.sort_values(KEY).itertuples(index=False):
        r = ref.loc[g.pilot_id]
        board = chess.Board(r.fen); move = board.parse_san(r.move_to_explain_san)
        evals = (r.eval_before_cp, r.eval_after_cp) if g.condition in ("B", "C") else ()
        eid = f"{slug(g.model)}|{g.condition}|r{g.repeat}|{g.pilot_id}"
        for i, c in enumerate(L.extract_claims(g.text)):
            k, p = L.classify(c["claim_text"]), L.precheck(c["claim_text"], board, move, r.kind, evals)
            out.append(dict(claim_id=f"{eid}|{i:02d}", explanation_id=eid, model=g.model, condition=g.condition, repeat=g.repeat, pilot_id=g.pilot_id,
                            **c, **k, **p))
    df = pd.DataFrame(out)
    df["route"] = df.apply(route, axis=1)
    return df


claims = run_extraction(gen[gen.usable])
again = run_extraction(gen[gen.usable])
selftest = hashlib.sha256(csv_bytes(claims)).hexdigest() == hashlib.sha256(csv_bytes(again)).hexdigest()
print(f"{len(claims)} claims from {claims.explanation_id.nunique()} explanations | determinism self-test passed: {selftest}")
if not selftest: warn("extraction is not deterministic")
no_claims = int(gen.usable.sum()) - claims.explanation_id.nunique()
if no_claims: warn(f"{no_claims} usable explanation(s) produced no claims")
print(claims.sample(min(12, len(claims)), random_state=config.SEED_ANNOTATION)[["model", "condition", "category", "auto_verdict", "claim_text"]].to_string(index=False, max_colwidth=70))

1061 claims from 119 explanations | determinism self-test passed: True
           model condition            category auto_verdict                                                             claim_text
     llama3.1:8b         C               Other      unknown     allows Black to potentially launch a counterattack on the kingside
     llama3.1:8b         C               Other      unknown                                  this move aims to create dynamic play
  deepseek-r1:8b         B               Other      unknown                                   Okay, let's break down Nd7 for Black
  deepseek-r1:8b         A  Tactical reasoning      unknown        disrupting its defensive or attacking functions in the position
  deepseek-r1:8b         B  Tactical reasoning      unknown                    This move completely ignores the threat to the king
phi4-mini:latest         C Board comprehension      unknown                                          Black is preparing to push g6
  deepseek-r

## 5. Summaries (what the professor asked to report: claim counts, strategic/unverifiable share, auto-checkable share)

In [6]:
cats = claims.groupby(["model", "condition", "category"]).size().unstack(fill_value=0).reindex(columns=list(L.CATEGORIES), fill_value=0).reset_index()
per_expl = claims.groupby(["model", "condition", "explanation_id"]).size().rename("n_claims").reset_index()
stats = per_expl.groupby(["model", "condition"]).n_claims.agg(explanations="count", mean="mean", sd="std", min="min", max="max").round(2).reset_index()
shares = claims.groupby(["model", "condition"]).agg(claims=("claim_id", "size"), hedged=("hedged", "mean"), strategic=("is_strategic", "mean"),
        review_category=("review_category", "mean"), mentions_engine=("mentions_engine", "mean")).round(3).reset_index()
routes = claims.groupby(["model", "condition", "route"]).size().unstack(fill_value=0).reset_index()
verd = claims.groupby(["model", "condition", "auto_verdict"]).size().unstack(fill_value=0).reset_index()
for name, t in (("categories", cats), ("claims per explanation", stats), ("shares", shares), ("verification route", routes), ("proposed verdicts", verd)):
    print(f"\n== {name} =="); print(t.to_string(index=False))


== categories ==
           model condition  Board comprehension  Legality  Quality evaluation  Tactical reasoning  Other
  deepseek-r1:8b         A                   19         1                   3                  26     11
  deepseek-r1:8b         B                   18         0                  13                  55     35
  deepseek-r1:8b         C                   11         0                  14                  48     32
     llama3.1:8b         A                   28         0                   4                  12     40
     llama3.1:8b         B                   23         0                  19                  11     42
     llama3.1:8b         C                   18         0                  10                   7     54
phi4-mini:latest         A                   14         0                   6                  21     46
phi4-mini:latest         B                   15         0                  20                  19     36
phi4-mini:latest         C           

## 6. Audit and annotation files

* `extraction_audit.csv`: a seeded sample of explanations next to their extracted claims - count missed / spurious / badly split claims by hand to measure the extractor before trusting it.
* `annotation_sheet_blind.csv`: all claims shuffled with `SEED_ANNOTATION`; no model, condition or pre-check visible. `annotation_key.csv` maps back. Claims that quote engine output (`mentions_engine`) can still reveal the condition.
* `annotation_second_subset.csv`: stratified (condition x category) `ANNOTATION_FRACTION` of the claims for the second annotator (rounded up per stratum).

In [7]:
rng = random.Random(config.SEED_ANNOTATION)
audit_ids = sorted(rng.sample(sorted(claims.explanation_id.unique()), min(AUDIT_N, claims.explanation_id.nunique())))
audit = []
for eid in audit_ids:
    gsel = gen[(gen.model.map(slug) == eid.split("|")[0]) & (gen.condition == eid.split("|")[1]) & (gen.pilot_id == eid.split("|")[3])
               & (gen.repeat == int(eid.split("|")[2][1:]))].iloc[0]
    items = claims[claims.explanation_id == eid].claim_text.tolist()
    audit.append(dict(explanation_id=eid, original_text=gsel.text, extracted="\n".join(f"{i + 1}. {t}" for i, t in enumerate(items)),
                      n_extracted=len(items), n_missed="", n_spurious="", n_badly_split="", notes=""))
audit = pd.DataFrame(audit)

order = sorted(claims.claim_id); random.Random(f"{config.SEED_ANNOTATION}:blind").shuffle(order)
blind_id = {cid: f"C{i + 1:04d}" for i, cid in enumerate(order)}
c2 = claims.assign(blind_id=claims.claim_id.map(blind_id)).sort_values("blind_id")
c2 = c2.merge(ref[["fen", "side_to_move", "move_to_explain_san"]], left_on="pilot_id", right_index=True)
sheet = c2[["blind_id", "fen", "side_to_move", "move_to_explain_san", "sentence", "claim_text"]].assign(
    verdict="", category_manual="", notes="")                      # verdict: correct / incorrect / unverifiable / uncertain
sub_ids = []
r2 = random.Random(f"{config.SEED_ANNOTATION}:second")
for _, g in claims.groupby(["condition", "category"]):
    sub_ids += r2.sample(sorted(g.claim_id), max(1, math.ceil(config.ANNOTATION_FRACTION * len(g))))
key = c2[["blind_id", "claim_id", "explanation_id", "model", "condition", "repeat", "pilot_id", "category", "category_rule", "auto_verdict",
          "auto_rule", "route", "mentions_engine"]].assign(second_annotator=c2.claim_id.isin(set(sub_ids)).values)
second = sheet[key.second_annotator.values]
print(f"annotation sheet: {len(sheet)} claims | second-annotator subset: {len(second)} ({len(second) / len(sheet):.0%})")

annotation sheet: 1061 claims | second-annotator subset: 218 (21%)


## 7. Write the outputs and the manifest

In [8]:
tables = {"claims_all.csv": claims, "generations_qc.csv": qc, "excluded_generations.csv": excluded, "summary_categories.csv": cats,
          "summary_claims_per_explanation.csv": stats, "summary_shares.csv": shares, "summary_routes.csv": routes, "summary_verdicts.csv": verd,
          "extraction_audit.csv": audit, "annotation_sheet_blind.csv": sheet, "annotation_key.csv": key, "annotation_second_subset.csv": second}
for name, df in tables.items(): (OUT / name).write_bytes(csv_bytes(df))
h = hashlib.sha256()
for name in sorted(tables): h.update(name.encode()); h.update((OUT / name).read_bytes())
manifest = dict(step="step4_claims", lib_version=L.LIB_VERSION, lib_sha256=sha256_file(PROJECT / "claims_lib.py"), notebook_source_sha256=notebook_code_sha256(),
    month=MONTH, models=MODELS, conditions=CONDITIONS, repeats=REPEATS, generations=len(gen), usable_generations=int(gen.usable.sum()), claims=len(claims),
    determinism_selftest_passed=selftest, flags=problems, pilot_manifest_sha256=sha256_file(P1 / "pilot_manifest.json"),
    step3_run_manifests_sha256={m: sha256_file(S3 / slug(m) / "run_manifest.json") for m in MODELS},
    seeds=dict(annotation=config.SEED_ANNOTATION), annotation_fraction=config.ANNOTATION_FRACTION, config=config.config_snapshot(),
    git_commit=git("rev-parse", "HEAD"), git_uncommitted_changes=bool(git("status", "--porcelain")), python=sys.version.split()[0], platform=platform.platform(),
    packages={p: metadata.version(p) for p in ("chess", "pandas")}, hashed_outputs=sorted(tables), results_sha256=h.hexdigest(),
    timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
(OUT / "step4_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"results_sha256: {manifest['results_sha256']} | flags: {len(problems)} | uncommitted changes: {manifest['git_uncommitted_changes']}")

results_sha256: 5e01887453bbbff18bee7007b3aa637f217b7bf795e8ae41864212ed2f58cc63 | flags: 2 | uncommitted changes: True
